# [투자전략] 계좌 인출 순서, 세후 잔액에 얼마나 영향을 줄까? (가상 데이터)

투자 조건을 입력하고 아래 코드 셀을 실행하세요. 연 상승률을 월 복리로
환산한 가상 자산으로, 원금성 재원과 과세 항목의 순서를 달리한 두 가지
인출 전략의 10년 세후 잔액과 납부세액을 비교합니다.

In [ ]:
# @title 인출 전략 비교 조건을 입력하고 실행하세요
연상승률_퍼센트 = 10.0  # @param {type:"number", min:-50, max:50, step:0.5}
초기준비금_억원 = 4.0  # @param {type:"number", min:0.01, step:0.1}
월실수령액_만원 = 500  # @param {type:"number", min:0, step:10}
투자기간_년 = 10  # @param {type:"integer", min:1, step:1}
분할매수기간_개월 = 12  # @param {type:"integer", min:1, max:24, step:1}

"""가상 투자자산 인출 순서 전략에 따른 세후 정기 인출 결과를 비교한다.

투자자산은 시장 데이터가 아니라 연 상승률을 매월 복리로 환산한 가상
가격을 사용한다(월 수익률 = (1+연상승률)^(1/12) - 1). 배당·환율·추적오차
개념이 없는 원화 단일자산으로, 매달 같은 비율로 상승한다고 가정한다.
미투자 준비금은 이자가 없는 원화 현금으로 두고, 매월 말 세후 실수령액을
먼저 인출한 뒤 12개월 동안 분할매수한다.

절세계좌는 1인 기준 ISA 누적 납입한도와 연금계좌 연간 한도를 활용한다.
ISA와 연금계좌에서도 같은 가상자산이 동일한 수익률을 낸다고 가정한다.
ISA는 납입원금 범위에서 세금 없이 중도인출하고, 원금 소진 후 해지할 때
전체 순이익을 정산한다. 3년 전 중도해지는 15.4%, 3년 이후 정상해지는
비과세 한도 공제 후 9.9%를 적용한다. 해지 잔액은 새 ISA에 즉시
재투자한다. 연금저축 납입금은 세액공제를 받지 않으며, 원금은 비과세로
먼저 인출하고 이후 연금 외 수령액에 16.5%를 적용한다. 미국계좌는 매년
250만원 양도소득 기본공제를 받는 공제분(2500만원 한도)과 그 초과분이
대기하는 과세분으로 나뉜다.

계좌 구조와 배분 규칙은 인출 순서에 관계없이 두 전략 모두 동일하며,
"미국계좌 과세분 → 국내일반계좌 → 연금저축 원금 → ISA 원금"까지는
공통이고, 그 뒤 연금저축 수익금·미국계좌 공제분·ISA 해지의 순서만
다르다. 상품 보수는 반영하지 않는다.
"""

from dataclasses import dataclass
import os
from pathlib import Path
from typing import Any, cast
from urllib.request import urlretrieve

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import font_manager
from matplotlib.axes import Axes
import pandas as pd
from IPython.display import HTML, Image, Markdown, display


def is_colab_runtime() -> bool:
    """현재 코드가 Google Colab에서 실행 중인지 확인한다."""
    return bool(os.environ.get("COLAB_RELEASE_TAG")) or Path("/content").exists()


IS_COLAB = is_colab_runtime()
OUTPUT_DIR = Path("/content/output") if IS_COLAB else Path("output")
WON_PER_EOK = 100_000_000
WON_PER_MANWON = 10_000
MONTHS_PER_YEAR = 12
ANNUAL_GROWTH_RATE = float(연상승률_퍼센트) / 100
MONTHLY_GROWTH_RATE = (1 + ANNUAL_GROWTH_RATE) ** (1 / MONTHS_PER_YEAR) - 1
INITIAL_RESERVE_KRW = float(초기준비금_억원) * WON_PER_EOK
MONTHLY_NET_WITHDRAWAL_KRW = float(월실수령액_만원) * WON_PER_MANWON
INVESTMENT_YEARS = int(투자기간_년)
DCA_MONTHS = int(분할매수기간_개월)
TOTAL_MONTHS = INVESTMENT_YEARS * MONTHS_PER_YEAR
INVESTOR_COUNT = 1
US_DEDUCTION_KRW = 2_500_000 * INVESTOR_COUNT
US_TAX_RATE = 0.22
GENERAL_TAX_RATE = 0.154
ISA_EXEMPTION_KRW = 2_000_000 * INVESTOR_COUNT
ISA_TAX_RATE = 0.099
PENSION_TAX_RATE = 0.165
ISA_ANNUAL_LIMIT_KRW = 20_000_000 * INVESTOR_COUNT
ISA_MAX_KRW = 100_000_000 * INVESTOR_COUNT
ISA_TERM_MONTHS = 36
PENSION_ANNUAL_LIMIT_KRW = 18_000_000 * INVESTOR_COUNT
US_CORE_KRW = 25_000_000 * INVESTOR_COUNT
GENERAL_MAX_KRW = 200_000_000 * INVESTOR_COUNT
INVESTOR_LABEL = "1인" if INVESTOR_COUNT == 1 else f"부부({INVESTOR_COUNT}인)"

# 두 전략은 "미국계좌 과세분 → 국내일반계좌 → 연금저축 원금 → ISA 원금"까지는
# 동일하고, 그 뒤 연금저축 수익금·미국계좌 공제분·ISA 해지의 순서만 다르다.
# ISA 해지로 생긴 재원의 재배분 순서에 미국계좌 공제분을 포함할지도 전략마다 다르다.
WITHDRAWAL_ORDERS: dict[str, tuple[str, ...]] = {
    "계좌순": (
        "us_excess", "general", "pension_principal", "pension_taxable",
        "isa_principal", "isa_close", "us_core",
    ),
    "과세이연": (
        "us_excess", "general", "pension_principal", "isa_principal",
        "us_core", "pension_taxable", "isa_close",
    ),
}
ISA_CLOSE_INCLUDES_US_CORE: dict[str, bool] = {
    "계좌순": False,
    "과세이연": True,
}
STRATEGIES = ("계좌순", "과세이연")
STRATEGY_COLORS = {
    "계좌순": "#2F7DD3",
    "과세이연": "#1FAE7A",
}
STRATEGY_LINESTYLES = {
    "계좌순": "solid",
    "과세이연": "solid",
}
BACKGROUND_COLOR = "#FFFFFF"
TEXT_COLOR = "#0B0B0B"
SECONDARY_TEXT_COLOR = "#64748B"
TICK_COLOR = "#777777"
GRID_COLOR = "#DEDCD6"
DEPLETION_COLOR = "#F06432"
REFERENCE_COLOR = "#F59E0B"
BRAND_SIZE = 13
TITLE_SIZE = 21
SUBTITLE_SIZE = 16
PANEL_TITLE_SIZE = 18
AXIS_TITLE_SIZE = 15
TICK_SIZE = 15
LEGEND_SIZE = 15
DATA_LABEL_SIZE = 14
REFERENCE_LABEL_SIZE = 14
FOOTNOTE_SIZE = 13

@dataclass
class Account:
    """계좌의 현금, QQQ 수량과 세금 계산용 원금을 보관한다."""
    cash_krw: float = 0.0
    shares: float = 0.0
    basis_krw: float = 0.0


@dataclass
class Portfolio:
    """절세계좌 전략의 계좌와 남은 공제액을 보관한다."""
    us: Account
    general: Account
    pension: Account
    isa: Account
    us_excess_cash_krw: float = 0.0
    us_deduction_remaining_krw: float = US_DEDUCTION_KRW
    isa_exemption_remaining_krw: float = ISA_EXEMPTION_KRW
    isa_principal_remaining_krw: float = 0.0
    isa_lifetime_contributed_krw: float = 0.0
    isa_age_months: int = ISA_TERM_MONTHS
    isa_closed: bool = False
    pension_principal_remaining_krw: float = 0.0
    pension_annual_contributed_krw: float = 0.0
    taxes_paid_krw: float = 0.0


def validate_parameters() -> None:
    """입력값을 검증한다."""
    if INITIAL_RESERVE_KRW <= 0 or INVESTMENT_YEARS <= 0:
        raise ValueError("초기 준비금과 투자기간은 0보다 커야 합니다.")
    if MONTHLY_NET_WITHDRAWAL_KRW < 0:
        raise ValueError("월 실수령액은 0원 이상이어야 합니다.")
    if not 1 <= DCA_MONTHS <= 24:
        raise ValueError("분할매수기간은 1~24개월이어야 합니다.")
    if ANNUAL_GROWTH_RATE <= -1:
        raise ValueError("연상승률은 -100%보다 커야 합니다.")


def build_virtual_market() -> pd.DataFrame:
    """연상승률을 월 복리로 환산한 가상 가격 시계열을 만든다.

    분할매수 거래월(0개월 차)부터 투자기간 마지막 달까지 경과 개월수를
    인덱스로 두고, 첫 달 가격을 1로 삼아 이후 모든 달이 (1+월수익률)^n
    배로 복리 성장하게 한다. 실제 달력이 없는 가상 시나리오이므로 날짜
    대신 경과 개월수(0부터 시작)를 시간축으로 쓴다.
    """
    month_numbers = range(TOTAL_MONTHS + 1)
    prices = [(1 + MONTHLY_GROWTH_RATE) ** n for n in month_numbers]
    return pd.DataFrame({"price_krw": prices}, index=pd.Index(month_numbers, name="month_number"))


def account_value(account: Account, price_krw: float) -> float:
    """계좌의 원화 평가액을 계산한다."""
    return account.cash_krw + account.shares * price_krw


def buy_from_cash(account: Account, requested_krw: float, price_krw: float) -> float:
    """계좌 현금으로 가상자산을 매수하고 실제 매수액을 반환한다."""
    purchase_krw = min(max(requested_krw, 0.0), account.cash_krw)
    account.cash_krw -= purchase_krw
    account.shares += purchase_krw / price_krw
    account.basis_krw += purchase_krw
    return purchase_krw


def buy_sequentially_from_accounts(
    portfolio: Portfolio, requested_krw: float, price_krw: float
) -> float:
    """미국계좌 공제분부터 절세계좌·일반계좌·미국계좌 과세분 순으로 매수한다."""
    remaining_krw = max(requested_krw, 0.0)
    purchased_krw = 0.0
    for name in ("us", "isa", "pension", "general"):
        if remaining_krw <= 0.5:
            break
        account = getattr(portfolio, name)
        amount_krw = buy_from_cash(account, remaining_krw, price_krw)
        purchased_krw += amount_krw
        remaining_krw -= amount_krw
    if remaining_krw > 0.5 and portfolio.us_excess_cash_krw > 0.0:
        amount_krw = min(remaining_krw, portfolio.us_excess_cash_krw)
        portfolio.us_excess_cash_krw -= amount_krw
        portfolio.us.shares += amount_krw / price_krw
        portfolio.us.basis_krw += amount_krw
        purchased_krw += amount_krw
    return purchased_krw


def sell_shares_for_net(
    account: Account, requested_net_krw: float, max_gross_krw: float,
    price_krw: float, tax_rate: float, exemption_krw: float,
) -> tuple[float, float, float]:
    """세금을 낸 뒤 목표 실수령액이 되도록 보유 수량을 매도한다."""
    share_value_krw = account.shares * price_krw
    gross_limit_krw = min(max(max_gross_krw, 0.0), share_value_krw)
    if requested_net_krw <= 0 or gross_limit_krw <= 0:
        return 0.0, 0.0, exemption_krw
    gain_ratio = (
        max(share_value_krw - account.basis_krw, 0.0) / share_value_krw
        if share_value_krw > 0 else 0.0
    )

    def net_from_gross(gross_krw: float) -> float:
        gain_krw = gross_krw * gain_ratio
        tax_krw = max(gain_krw - exemption_krw, 0.0) * tax_rate
        return gross_krw - tax_krw

    if gain_ratio <= 0 or requested_net_krw * gain_ratio <= exemption_krw:
        gross_needed_krw = requested_net_krw
    else:
        gross_needed_krw = (
            requested_net_krw - tax_rate * exemption_krw
        ) / (1 - tax_rate * gain_ratio)
    gross_sale_krw = min(max(gross_needed_krw, 0.0), gross_limit_krw)
    realized_gain_krw = gross_sale_krw * gain_ratio
    tax_krw = max(realized_gain_krw - exemption_krw, 0.0) * tax_rate
    net_krw = gross_sale_krw - tax_krw
    sold_fraction = gross_sale_krw / share_value_krw
    account.shares *= max(1 - sold_fraction, 0.0)
    account.basis_krw *= max(1 - sold_fraction, 0.0)
    exemption_krw = max(exemption_krw - realized_gain_krw, 0.0)
    return min(net_krw, net_from_gross(gross_limit_krw)), tax_krw, exemption_krw


def withdraw_net_from_account(
    account: Account, requested_net_krw: float, price_krw: float,
    tax_rate: float, exemption_krw: float = 0.0, floor_krw: float = 0.0,
) -> tuple[float, float, float]:
    """계좌 평가액 하한을 지키며 세후 실수령액을 인출한다."""
    available_gross_krw = max(account_value(account, price_krw) - floor_krw, 0.0)
    cash_withdrawal_krw = min(account.cash_krw, requested_net_krw, available_gross_krw)
    account.cash_krw -= cash_withdrawal_krw
    remaining_net_krw = requested_net_krw - cash_withdrawal_krw
    remaining_gross_krw = available_gross_krw - cash_withdrawal_krw
    sale_net_krw, tax_krw, exemption_krw = sell_shares_for_net(
        account, remaining_net_krw, remaining_gross_krw,
        price_krw, tax_rate, exemption_krw,
    )
    return cash_withdrawal_krw + sale_net_krw, tax_krw, exemption_krw


def withdraw_from_us_excess(
    portfolio: Portfolio, requested_net_krw: float
) -> tuple[float, float]:
    """미국계좌 과세분 예수금에서 인출한다. 공제분 계좌는 별개 계좌로 취급해 건드리지 않는다."""
    cash_krw = min(portfolio.us_excess_cash_krw, max(requested_net_krw, 0.0))
    portfolio.us_excess_cash_krw -= cash_krw
    return cash_krw, 0.0


def withdraw_principal_tax_free(
    account: Account, principal_remaining_krw: float, requested_krw: float,
    price_krw: float,
) -> tuple[float, float]:
    """계좌 납입원금 범위에서 세금 없이 중도인출하고, (인출액, 남은 원금)을 반환한다."""
    available_krw = account_value(account, price_krw)
    withdrawal_krw = min(max(requested_krw, 0.0), principal_remaining_krw, available_krw)
    cash_krw = min(account.cash_krw, withdrawal_krw)
    account.cash_krw -= cash_krw
    sale_krw = withdrawal_krw - cash_krw
    if sale_krw > 0.0:
        share_value_krw = account.shares * price_krw
        sold_fraction = min(sale_krw / share_value_krw, 1.0)
        account.shares *= max(1 - sold_fraction, 0.0)
        account.basis_krw *= max(1 - sold_fraction, 0.0)
    return withdrawal_krw, principal_remaining_krw - withdrawal_krw


def withdraw_isa_principal(
    portfolio: Portfolio, requested_krw: float, price_krw: float
) -> float:
    """ISA 납입원금 범위에서 세금 없이 중도인출한다."""
    if portfolio.isa_closed:
        return 0.0
    withdrawal_krw, portfolio.isa_principal_remaining_krw = withdraw_principal_tax_free(
        portfolio.isa, portfolio.isa_principal_remaining_krw, requested_krw, price_krw,
    )
    if account_value(portfolio.isa, price_krw) <= 0.5:
        portfolio.isa = Account()
        portfolio.isa_principal_remaining_krw = 0.0
        portfolio.isa_closed = True
    return withdrawal_krw


def withdraw_pension_principal(
    portfolio: Portfolio, requested_krw: float, price_krw: float
) -> float:
    """세액공제받지 않은 연금저축 원금을 세금 없이 인출한다."""
    withdrawal_krw, portfolio.pension_principal_remaining_krw = withdraw_principal_tax_free(
        portfolio.pension, portfolio.pension_principal_remaining_krw, requested_krw, price_krw,
    )
    return withdrawal_krw


def withdraw_taxable_pension(
    portfolio: Portfolio, requested_net_krw: float, price_krw: float
) -> tuple[float, float]:
    """연금저축 수익금을 연금 외 수령하고 16.5%를 원천징수한다."""
    available_gross_krw = account_value(portfolio.pension, price_krw)
    gross_needed_krw = max(requested_net_krw, 0.0) / (1 - PENSION_TAX_RATE)
    gross_withdrawal_krw = min(gross_needed_krw, available_gross_krw)
    cash_krw = min(portfolio.pension.cash_krw, gross_withdrawal_krw)
    portfolio.pension.cash_krw -= cash_krw
    sale_krw = gross_withdrawal_krw - cash_krw
    if sale_krw > 0.0:
        share_value_krw = portfolio.pension.shares * price_krw
        sold_fraction = min(sale_krw / share_value_krw, 1.0)
        portfolio.pension.shares *= max(1 - sold_fraction, 0.0)
        portfolio.pension.basis_krw *= max(1 - sold_fraction, 0.0)
    tax_krw = gross_withdrawal_krw * PENSION_TAX_RATE
    return gross_withdrawal_krw - tax_krw, tax_krw


def close_isa(
    portfolio: Portfolio, requested_net_krw: float, price_krw: float,
    include_us_core: bool,
) -> tuple[float, float]:
    """ISA를 해지하고 생활비 차감 후 남은 재원을 재배분한다.

    include_us_core가 참이면 미국계좌 공제분(2500만원까지)을 재배분 1순위로 채운 뒤
    ISA→연금저축→일반계좌→미국계좌 과세분 순으로 진행하고, 거짓이면 공제분 없이
    ISA→연금저축→일반계좌→미국계좌 과세분 순으로 진행한다.
    """
    if portfolio.isa_closed:
        return 0.0, 0.0
    value_krw = account_value(portfolio.isa, price_krw)
    gain_krw = max(value_krw - portfolio.isa_principal_remaining_krw, 0.0)
    is_mature = portfolio.isa_age_months >= ISA_TERM_MONTHS
    tax_krw = (
        max(gain_krw - portfolio.isa_exemption_remaining_krw, 0.0) * ISA_TAX_RATE
        if is_mature
        else gain_krw * GENERAL_TAX_RATE
    )
    proceeds_krw = max(value_krw - tax_krw, 0.0)
    withdrawal_krw = min(max(requested_net_krw, 0.0), proceeds_krw)
    remaining_krw = proceeds_krw - withdrawal_krw

    us_core_krw = 0.0
    if include_us_core:
        us_core_room_krw = max(US_CORE_KRW - account_value(portfolio.us, price_krw), 0.0)
        us_core_krw = min(remaining_krw, us_core_room_krw)
        remaining_krw -= us_core_krw

    new_isa_krw = min(remaining_krw, ISA_ANNUAL_LIMIT_KRW)
    remaining_krw -= new_isa_krw

    pension_room_krw = max(
        PENSION_ANNUAL_LIMIT_KRW - portfolio.pension_annual_contributed_krw, 0.0
    )
    pension_contribution_krw = min(remaining_krw, pension_room_krw)
    remaining_krw -= pension_contribution_krw

    general_room_krw = max(GENERAL_MAX_KRW - account_value(portfolio.general, price_krw), 0.0)
    general_cash_krw = min(remaining_krw, general_room_krw)
    remaining_krw -= general_cash_krw

    us_excess_krw = remaining_krw

    if us_core_krw > 0.0:
        portfolio.us.cash_krw += us_core_krw
        buy_from_cash(portfolio.us, us_core_krw, price_krw)
    if new_isa_krw > 0.0:
        portfolio.isa = Account(
            shares=new_isa_krw / (price_krw),
            basis_krw=new_isa_krw,
        )
        portfolio.isa_principal_remaining_krw = new_isa_krw
        portfolio.isa_lifetime_contributed_krw += new_isa_krw
        portfolio.isa_exemption_remaining_krw = ISA_EXEMPTION_KRW
        portfolio.isa_age_months = 0
        portfolio.isa_closed = False
    else:
        portfolio.isa = Account()
        portfolio.isa_principal_remaining_krw = 0.0
        portfolio.isa_exemption_remaining_krw = 0.0
        portfolio.isa_closed = True
    if pension_contribution_krw > 0.0:
        portfolio.pension.cash_krw += pension_contribution_krw
        buy_from_cash(portfolio.pension, pension_contribution_krw, price_krw)
        portfolio.pension_principal_remaining_krw += pension_contribution_krw
        portfolio.pension_annual_contributed_krw += pension_contribution_krw
    if general_cash_krw > 0.0:
        portfolio.general.cash_krw += general_cash_krw
        buy_from_cash(portfolio.general, general_cash_krw, price_krw)
    if us_excess_krw > 0.0:
        portfolio.us_excess_cash_krw += us_excess_krw
    return withdrawal_krw, tax_krw


def initialize_tax_portfolio() -> Portfolio:
    """가구원 수 기준 절세 우선순위로 초기 준비금을 배분한다."""
    remaining_krw = INITIAL_RESERVE_KRW
    allocations: dict[str, float] = {}
    for name, target_krw in (
        ("us", US_CORE_KRW),
        ("isa", ISA_MAX_KRW),
        ("pension", PENSION_ANNUAL_LIMIT_KRW),
    ):
        allocations[name] = min(remaining_krw, target_krw)
        remaining_krw -= allocations[name]
    allocations["general"] = min(remaining_krw, GENERAL_MAX_KRW)
    remaining_krw -= allocations["general"]
    allocations["us_excess"] = remaining_krw
    return Portfolio(
        us=Account(cash_krw=allocations["us"]),
        general=Account(cash_krw=allocations["general"]),
        pension=Account(cash_krw=allocations["pension"]),
        isa=Account(cash_krw=allocations["isa"]),
        us_excess_cash_krw=allocations["us_excess"],
        isa_principal_remaining_krw=allocations["isa"],
        isa_lifetime_contributed_krw=allocations["isa"],
        pension_principal_remaining_krw=allocations["pension"],
        pension_annual_contributed_krw=allocations["pension"],
    )


def portfolio_value(portfolio: Portfolio, price_krw: float) -> float:
    """절세계좌 포트폴리오의 총평가액을 계산한다."""
    return portfolio.us_excess_cash_krw + sum(
        account_value(account, price_krw)
        for account in (portfolio.us, portfolio.general, portfolio.pension, portfolio.isa)
    )


def withdraw_from_tax_portfolio(
    portfolio: Portfolio, requested_net_krw: float, price_krw: float,
    withdrawal_order: tuple[str, ...], isa_close_includes_us_core: bool,
) -> tuple[float, float]:
    """전략별 계좌 순서에 따라 세후 실수령액을 마련한다. 한 바퀴를 돌고도 부족하면 재배분된
    자금(ISA 해지로 생긴 재원 등)을 반영해 순서를 한 번 더 순회한다."""
    remaining_net_krw = requested_net_krw
    total_tax_krw = 0.0
    made_progress = True
    while remaining_net_krw > 0.5 and made_progress:
        made_progress = False
        for source in withdrawal_order:
            if remaining_net_krw <= 0.5:
                break
            if source == "us_excess":
                net_krw, tax_krw = withdraw_from_us_excess(portfolio, remaining_net_krw)
            elif source == "pension_principal":
                net_krw = withdraw_pension_principal(
                    portfolio, remaining_net_krw, price_krw
                )
                tax_krw = 0.0
            elif source == "isa_principal":
                net_krw = withdraw_isa_principal(
                    portfolio, remaining_net_krw, price_krw
                )
                tax_krw = 0.0
            elif source == "us_core":
                net_krw, tax_krw, exemption_krw = withdraw_net_from_account(
                    portfolio.us, remaining_net_krw, price_krw,
                    US_TAX_RATE, portfolio.us_deduction_remaining_krw,
                )
                portfolio.us_deduction_remaining_krw = exemption_krw
            elif source == "pension_taxable":
                net_krw, tax_krw = withdraw_taxable_pension(
                    portfolio, remaining_net_krw, price_krw
                )
            elif source == "isa_close":
                net_krw = 0.0
                tax_krw = 0.0
                if not portfolio.isa_closed:
                    net_krw, tax_krw = close_isa(
                        portfolio, remaining_net_krw, price_krw,
                        isa_close_includes_us_core,
                    )
            else:
                account = getattr(portfolio, source)
                tax_rate = {"general": GENERAL_TAX_RATE}[source]
                net_krw, tax_krw, exemption_krw = withdraw_net_from_account(
                    account, remaining_net_krw, price_krw, tax_rate
                )
            remaining_net_krw -= net_krw
            total_tax_krw += tax_krw
            if net_krw > 0.5:
                made_progress = True
    portfolio.taxes_paid_krw += total_tax_krw
    return requested_net_krw - max(remaining_net_krw, 0.0), total_tax_krw


def redistribute_us_excess_january(
    portfolio: Portfolio, price_krw: float
) -> None:
    """1월에 미국계좌 과세분(전년 12월 리밸런싱 이동금액 포함)을 ISA→연금저축→일반계좌 순으로 채워 넣는다.

    ISA는 평생 누적 납입한도를 쓰므로, 중도인출로 원금이 줄어도 납입 여유가 되살아나지
    않는다. 따라서 여유는 인출 가능 원금이 아니라 이제까지 실제로 납입한 누적액 기준으로 잰다.
    """
    isa_room_krw = max(ISA_MAX_KRW - portfolio.isa_lifetime_contributed_krw, 0.0)
    isa_contribution_krw = min(portfolio.us_excess_cash_krw, isa_room_krw)
    if isa_contribution_krw > 0.5:
        portfolio.us_excess_cash_krw -= isa_contribution_krw
        portfolio.isa.cash_krw += isa_contribution_krw
        buy_from_cash(portfolio.isa, isa_contribution_krw, price_krw)
        portfolio.isa_principal_remaining_krw += isa_contribution_krw
        portfolio.isa_lifetime_contributed_krw += isa_contribution_krw

    pension_room_krw = max(
        PENSION_ANNUAL_LIMIT_KRW - portfolio.pension_annual_contributed_krw, 0.0
    )
    pension_contribution_krw = min(portfolio.us_excess_cash_krw, pension_room_krw)
    if pension_contribution_krw > 0.5:
        portfolio.us_excess_cash_krw -= pension_contribution_krw
        portfolio.pension.cash_krw += pension_contribution_krw
        buy_from_cash(portfolio.pension, pension_contribution_krw, price_krw)
        portfolio.pension_principal_remaining_krw += pension_contribution_krw
        portfolio.pension_annual_contributed_krw += pension_contribution_krw

    general_contribution_krw = portfolio.us_excess_cash_krw
    if general_contribution_krw > 0.5:
        portfolio.us_excess_cash_krw -= general_contribution_krw
        portfolio.general.cash_krw += general_contribution_krw
        buy_from_cash(portfolio.general, general_contribution_krw, price_krw)


def fund_annual_pension(
    portfolio: Portfolio, price_krw: float
) -> float:
    """미국계좌 과세분과 일반계좌에서 연금저축 납입 재원을 마련한다(이미 채워진 한도는 제외)."""
    room_krw = max(
        PENSION_ANNUAL_LIMIT_KRW - portfolio.pension_annual_contributed_krw, 0.0
    )
    remaining_krw = room_krw
    net_krw, tax_krw = withdraw_from_us_excess(portfolio, remaining_krw)
    portfolio.taxes_paid_krw += tax_krw
    portfolio.pension.cash_krw += net_krw
    remaining_krw -= net_krw
    if remaining_krw > 0.5:
        gen_net_krw, gen_tax_krw, _ = withdraw_net_from_account(
            portfolio.general, remaining_krw, price_krw, GENERAL_TAX_RATE
        )
        portfolio.taxes_paid_krw += gen_tax_krw
        portfolio.pension.cash_krw += gen_net_krw
        remaining_krw -= gen_net_krw
    contributed_krw = room_krw - max(remaining_krw, 0.0)
    buy_from_cash(portfolio.pension, contributed_krw, price_krw)
    portfolio.pension_principal_remaining_krw += contributed_krw
    portfolio.pension_annual_contributed_krw += contributed_krw
    return contributed_krw


def rebalance_us_core_december(
    portfolio: Portfolio, price_krw: float
) -> float:
    """12월에 미국계좌 공제분의 2500만원 초과분 전체를 매도해 과세분으로 이동한다.

    남은 기본공제 한도까지는 비과세, 그 초과분에는 22% 세금을 매긴 뒤 세후 금액을 과세분
    예수금으로 옮긴다. 초과분이 없으면(잔액이 2500만원 이하) 아무 것도 하지 않는다.
    """
    account = portfolio.us
    core_value_krw = account_value(account, price_krw)
    excess_krw = max(core_value_krw - US_CORE_KRW, 0.0)
    if excess_krw <= 0.0:
        return 0.0
    gain_ratio = (
        max(core_value_krw - account.basis_krw, 0.0) / core_value_krw
        if core_value_krw > 0 else 0.0
    )
    realized_gain_krw = excess_krw * gain_ratio
    exemption_krw = portfolio.us_deduction_remaining_krw
    tax_krw = max(realized_gain_krw - exemption_krw, 0.0) * US_TAX_RATE
    portfolio.us_deduction_remaining_krw = max(exemption_krw - realized_gain_krw, 0.0)
    net_krw = excess_krw - tax_krw
    sold_fraction = excess_krw / core_value_krw
    account.shares *= max(1 - sold_fraction, 0.0)
    account.basis_krw *= max(1 - sold_fraction, 0.0)
    portfolio.us_excess_cash_krw += net_krw
    portfolio.taxes_paid_krw += tax_krw
    return net_krw


def liquidation_value(
    account: Account, price_krw: float, tax_rate: float, exemption_krw: float
) -> float:
    """계좌를 전액 청산했을 때의 세후 금액을 계산한다."""
    invested_value_krw = account.shares * price_krw
    gain_krw = max(invested_value_krw - account.basis_krw, 0.0)
    tax_krw = max(gain_krw - exemption_krw, 0.0) * tax_rate
    return account.cash_krw + invested_value_krw - tax_krw


def isa_liquidation_value(
    portfolio: Portfolio, price_krw: float
) -> float:
    """ISA 보유기간에 따라 정상해지 또는 중도해지 세금을 적용한다."""
    if portfolio.isa_closed:
        return 0.0
    value_krw = account_value(portfolio.isa, price_krw)
    gain_krw = max(value_krw - portfolio.isa_principal_remaining_krw, 0.0)
    tax_krw = (
        max(gain_krw - portfolio.isa_exemption_remaining_krw, 0.0) * ISA_TAX_RATE
        if portfolio.isa_age_months >= ISA_TERM_MONTHS
        else gain_krw * GENERAL_TAX_RATE
    )
    return max(value_krw - tax_krw, 0.0)


def pension_liquidation_value(
    portfolio: Portfolio, price_krw: float
) -> float:
    """연금저축 원금을 제외한 연금 외 수령액에 16.5%를 적용한다."""
    value_krw = account_value(portfolio.pension, price_krw)
    taxable_krw = max(value_krw - portfolio.pension_principal_remaining_krw, 0.0)
    return max(value_krw - taxable_krw * PENSION_TAX_RATE, 0.0)


def simulate_withdrawal_strategy(
    market: pd.DataFrame, strategy: str
) -> tuple[dict[str, Any], pd.DataFrame]:
    """가상 시장 데이터로 한 인출 순서 전략의 월별 인출 과정을 계산한다."""
    withdrawal_order = WITHDRAWAL_ORDERS[strategy]
    isa_close_includes_us_core = ISA_CLOSE_INCLUDES_US_CORE[strategy]
    portfolio = initialize_tax_portfolio()
    monthly_purchase_krw = INITIAL_RESERVE_KRW / DCA_MONTHS
    total_withdrawn_krw = 0.0
    depletion_month_number = None
    rows = []
    for month_number, values in market.iterrows():
        price_krw = float(values["price_krw"])
        if month_number > 0 and not portfolio.isa_closed:
            portfolio.isa_age_months += 1
        if month_number > 0 and month_number % MONTHS_PER_YEAR == 0:
            portfolio.us_deduction_remaining_krw = US_DEDUCTION_KRW
            portfolio.pension_annual_contributed_krw = 0.0
            redistribute_us_excess_january(portfolio, price_krw)
            fund_annual_pension(portfolio, price_krw)
        actual_withdrawal_krw = 0.0
        tax_krw = 0.0
        if month_number > 0:
            actual_withdrawal_krw, tax_krw = withdraw_from_tax_portfolio(
                portfolio, MONTHLY_NET_WITHDRAWAL_KRW, price_krw,
                withdrawal_order, isa_close_includes_us_core,
            )
            total_withdrawn_krw += actual_withdrawal_krw
        if month_number < DCA_MONTHS:
            buy_sequentially_from_accounts(
                portfolio, monthly_purchase_krw, price_krw
            )
        if month_number > 0 and month_number % MONTHS_PER_YEAR == 0:
            rebalance_us_core_december(portfolio, price_krw)
        ending_balance_krw = portfolio_value(portfolio, price_krw)
        if month_number > 0 and (
            actual_withdrawal_krw + 0.5 < MONTHLY_NET_WITHDRAWAL_KRW
            or ending_balance_krw <= 0.5
        ):
            depletion_month_number = month_number
        us_core_value_krw = account_value(portfolio.us, price_krw)
        us_core_principal_krw = min(portfolio.us.cash_krw + portfolio.us.basis_krw, us_core_value_krw)
        isa_value_krw = account_value(portfolio.isa, price_krw)
        isa_principal_krw = min(portfolio.isa_principal_remaining_krw, isa_value_krw)
        pension_value_krw = account_value(portfolio.pension, price_krw)
        pension_principal_krw = min(portfolio.pension_principal_remaining_krw, pension_value_krw)
        rows.append({
            "strategy": strategy, "month_number": month_number,
            "actual_withdrawal_krw": actual_withdrawal_krw,
            "tax_paid_krw": tax_krw, "ending_balance_krw": ending_balance_krw,
            "us_core_value_krw": us_core_value_krw,
            "us_core_principal_krw": us_core_principal_krw,
            "us_core_gain_krw": max(us_core_value_krw - us_core_principal_krw, 0.0),
            "isa_principal_krw": isa_principal_krw,
            "isa_gain_krw": max(isa_value_krw - isa_principal_krw, 0.0),
            "pension_principal_krw": pension_principal_krw,
            "pension_gain_krw": max(pension_value_krw - pension_principal_krw, 0.0),
            "general_krw": account_value(portfolio.general, price_krw),
            "us_excess_krw": portfolio.us_excess_cash_krw,
        })
        if depletion_month_number is not None:
            break
    if depletion_month_number is None:
        final_price_krw = float(market.iloc[-1]["price_krw"])
        final_balance_krw = portfolio.us_excess_cash_krw + sum((
            liquidation_value(
                portfolio.us, final_price_krw,
                US_TAX_RATE, portfolio.us_deduction_remaining_krw,
            ),
            liquidation_value(
                portfolio.general, final_price_krw, GENERAL_TAX_RATE, 0.0
            ),
            pension_liquidation_value(portfolio, final_price_krw),
            isa_liquidation_value(portfolio, final_price_krw),
        ))
    else:
        final_balance_krw = 0.0
    summary = {
        "strategy": strategy,
        "depleted_within_period": depletion_month_number is not None,
        "depletion_month_number": depletion_month_number if depletion_month_number is not None else -1,
        "months_survived": max(len(rows) - 1, 0),
        "total_withdrawn_krw": total_withdrawn_krw,
        "taxes_paid_krw": portfolio.taxes_paid_krw,
        "ending_balance_krw": final_balance_krw,
        "pre_liquidation_balance_krw": float(rows[-1]["ending_balance_krw"]),
    }
    return summary, pd.DataFrame(rows)


def calculate_results(market: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """인출 전략별 요약·월별 결과를 계산한다."""
    summaries = []
    details = []
    for strategy in STRATEGIES:
        summary, detail = simulate_withdrawal_strategy(market, strategy)
        summaries.append(summary)
        details.append(detail)
    return pd.DataFrame(summaries), pd.concat(details, ignore_index=True)


def format_duration(months: int) -> str:
    """개월 수를 연·개월 문자열로 표시한다."""
    years, remaining_months = divmod(months, MONTHS_PER_YEAR)
    if remaining_months == 0:
        return f"{years}년"
    return f"{years}년 {remaining_months}개월" if years else f"{remaining_months}개월"


def format_korean_won(amount_krw: float) -> str:
    """원화 금액을 억원·만원 단위로 표시한다."""
    rounded_manwon = int(round(max(amount_krw, 0.0) / WON_PER_MANWON))
    eok, manwon = divmod(rounded_manwon, 10_000)
    if eok and manwon:
        return f"{eok:,}억 {manwon:,}만원"
    if eok:
        return f"{eok:,}억원"
    return f"{manwon:,}만원" if manwon else "0원"


def evaluated_final_balance(row: Any) -> float:
    """완주 잔액 또는 계획 대비 누적 인출 부족액을 반환한다."""
    if bool(row.depleted_within_period):
        planned_withdrawal_krw = MONTHLY_NET_WITHDRAWAL_KRW * TOTAL_MONTHS
        return -(planned_withdrawal_krw - float(row.total_withdrawn_krw))
    return float(row.ending_balance_krw)


def result_table_html(summary: pd.DataFrame) -> str:
    """가상 시나리오의 전략별 결과를 HTML 표로 만든다."""
    def format_result_value(value_krw: float) -> str:
        formatted = format_korean_won(abs(value_krw))
        if value_krw < 0:
            return f'<span class="depleted-value">-{formatted}</span>'
        return formatted

    rows: list[dict[str, Any]] = []
    for strategy in STRATEGIES:
        result = cast(Any, summary.loc[summary["strategy"] == strategy].iloc[0])
        depleted = bool(result.depleted_within_period)
        value_krw = evaluated_final_balance(result)
        rows.append({
            "전략": strategy,
            "결과": "고갈" if depleted else "생존",
            "생존 기간": format_duration(int(result.months_survived)),
            "최종 잔액": format_result_value(value_krw),
            "총 인출액": format_korean_won(float(result.total_withdrawn_krw)),
            "납부 세액": format_korean_won(float(result.taxes_paid_krw)),
        })
    table = pd.DataFrame(rows).to_html(
        index=False, border=0, classes="result-table", escape=False
    )
    subtitle = (
        f"연 {연상승률_퍼센트:g}% 상승(월 복리) 가정 · "
        f"초기 준비금 {format_korean_won(INITIAL_RESERVE_KRW).replace('억원', '억 원')} · "
        f"월 {format_korean_won(MONTHLY_NET_WITHDRAWAL_KRW).replace('만원', '만 원')} 인출 · "
        f"{INVESTOR_LABEL} 기준"
    )
    css = """
    <style>
    .result-table-section {max-width:820px; margin:28px 0;}
    .result-table-brand {margin:0 0 5px; color:#64748b; font:400 13px
      Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;}
    .result-table-title {margin:0 0 4px; color:#0f172a; font:700 20px
      Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;}
    .result-table-caption {margin:0 0 14px; color:#64748b; font:400 13px
      Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;}
    .result-table-wrap {overflow:hidden; border:1px solid #f0f2f5;
      border-radius:12px; background:white;}
    .result-table {width:100%; table-layout:fixed; border-collapse:separate;
      border-spacing:0; color:#1e293b; font:400 13px
      Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
      font-variant-numeric:tabular-nums;}
    .result-table th {padding:11px 8px; background:#2b4a75; color:white;
      border-bottom:2px solid #7fb3d5; font-weight:700; text-align:center;
      white-space:nowrap;}
    .result-table td {padding:11px 8px; border-bottom:1px solid #f0f2f5;
      background:white; text-align:right; white-space:nowrap;}
    .result-table td:first-child {text-align:center; font-weight:600;}
    .result-table tbody tr:nth-child(even) td {background:#fafbfc;}
    .result-table tbody tr:nth-last-child(-n+3) td {background:#f1f4f8;
      font-weight:700;}
    .result-table tbody tr:nth-last-child(-n+3) td:first-child {text-align:left;}
    .result-table tbody tr:nth-last-child(3) td {border-top:2px solid #cbd5e1;}
    .result-table tbody tr:last-child td {border-bottom:0;}
    .result-table .depleted-value {color:#f06432;}
    .result-table tbody tr:hover td {background:#eff6ff;}
    .result-table-note {margin:10px 0 0; color:#64748b; font:400 13px/1.5
      Pretendard,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;}
    </style>
    """
    heading = (
        '<div class="result-table-section">'
        '<div class="result-table-brand">대도시 연구실</div>'
        f'<div class="result-table-title">인출 순서 {INVESTMENT_YEARS}년 뒤 '
        '최종 잔액 | 전략별 결과</div>'
        f'<div class="result-table-caption">{subtitle}</div>'
        f'<div class="result-table-wrap">{table}</div>'
        '<div class="result-table-note">※ 계좌순: 연금저축 수익금·미국계좌 공제분·ISA '
        '해지 순으로 인출하고 ISA 해지 재원에 미국계좌 공제분 제외 · 과세이연: 미국계좌 '
        '공제분을 연금저축 수익금보다 먼저 인출하고 ISA 해지 재원에 미국계좌 공제분 포함</div></div>'
    )
    return css + heading


FONT_CANDIDATES = ("Pretendard", "Apple SD Gothic Neo", "Noto Sans CJK KR", "Malgun Gothic")
COLAB_FONT_PATH = Path("/content/.fonts/Pretendard-Regular.otf")
COLAB_FONT_URL = ("https://raw.githubusercontent.com/orioncactus/pretendard/main/"
                  "packages/pretendard/dist/public/static/Pretendard-Regular.otf")


def configure_korean_font() -> None:
    """사용 가능한 한글 글꼴을 Matplotlib에 설정한다."""
    installed = {font.name for font in font_manager.fontManager.ttflist}
    font_name = next((name for name in FONT_CANDIDATES if name in installed), None)
    if font_name is None and IS_COLAB:
        COLAB_FONT_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not COLAB_FONT_PATH.exists():
            urlretrieve(COLAB_FONT_URL, COLAB_FONT_PATH)
        font_manager.fontManager.addfont(COLAB_FONT_PATH)
        font_name = font_manager.FontProperties(fname=COLAB_FONT_PATH).get_name()
    if font_name:
        plt.rcParams["font.family"] = font_name
    plt.rcParams["axes.unicode_minus"] = False



def draw_balance_flow_lines(ax: Axes, flow: pd.DataFrame, results: pd.DataFrame) -> None:
    """전략별 잔액 곡선을 선 스타일로 구분해 그리고, 마지막 결과를 우측 상단에 정렬해 표기한다."""
    for strategy in STRATEGIES:
        rows = flow.loc[flow["strategy"] == strategy].sort_values("month_number")
        if rows.empty:
            continue
        elapsed_years = rows["month_number"] / MONTHS_PER_YEAR
        balances = rows["ending_balance_krw"] / WON_PER_EOK
        ax.plot(
            elapsed_years, balances, color=STRATEGY_COLORS[strategy],
            linewidth=2.6, linestyle=STRATEGY_LINESTYLES[strategy],
            label=strategy, zorder=3,
        )
        last = cast(Any, rows.iloc[-1])
        result = cast(Any, results.loc[results["strategy"] == strategy].iloc[0])
        depleted = bool(result.depleted_within_period)
        if depleted:
            ax.scatter(last.month_number / MONTHS_PER_YEAR, 0, s=60, color=STRATEGY_COLORS[strategy], zorder=4)

    for legend_index, strategy in enumerate(STRATEGIES):
        rows = flow.loc[flow["strategy"] == strategy].sort_values("month_number")
        result = cast(Any, results.loc[results["strategy"] == strategy].iloc[0])
        depleted = bool(result.depleted_within_period)
        if depleted:
            last_withdrawal_krw = float(rows.iloc[-1].actual_withdrawal_krw)
            last_withdrawal_manwon = round(last_withdrawal_krw / WON_PER_MANWON)
            label = (
                f"{strategy}: {format_duration(int(result.months_survived))} 후 고갈 "
                f"(마지막 지급액 {last_withdrawal_manwon:,}만원)"
            )
        else:
            label = f"{strategy}: {format_korean_won(float(result.ending_balance_krw))}"
        ax.text(
            0.98, 0.83 - legend_index * 0.075, label,
            transform=ax.transAxes, ha="right", va="top",
            fontsize=DATA_LABEL_SIZE, fontweight="bold",
            color=STRATEGY_COLORS[strategy],
        )


def save_balance_flow_chart(
    summary: pd.DataFrame, detail: pd.DataFrame, output_path: Path
) -> None:
    """인출 전략별 월말 잔액 흐름을 저장한다.

    두 전략은 원금성 재원의 인출 순서가 같아 초반 잔액 곡선이 대부분 겹치고,
    공제분·연금 수익금·ISA 해지 순서가 갈리는 후반부에서 벌어진다.
    """
    flow = detail.copy()
    results = summary.copy()
    if flow.empty or len(results) != len(STRATEGIES):
        raise RuntimeError("잔액 흐름 그래프에는 전략별 결과가 필요합니다.")

    configure_korean_font()
    fig, ax = plt.subplots(figsize=(11.5, 7.2))
    fig.patch.set_facecolor(BACKGROUND_COLOR)
    ax.set_facecolor(BACKGROUND_COLOR)

    draw_balance_flow_lines(ax, flow, results)

    reference = INITIAL_RESERVE_KRW / WON_PER_EOK
    ax.axhline(
        reference, color=REFERENCE_COLOR, linewidth=1.5,
        linestyle=(0, (2, 3)), alpha=0.7, zorder=1,
    )
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
    ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:.0f}년"))
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:g}"))
    ax.tick_params(axis="both", colors=TICK_COLOR, labelsize=TICK_SIZE, length=0)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.9, zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.text(
        0.01, reference, f"시작 준비금 {format_korean_won(INITIAL_RESERVE_KRW)}",
        transform=ax.get_yaxis_transform(), ha="left", va="bottom",
        fontsize=REFERENCE_LABEL_SIZE, color=REFERENCE_COLOR,
        bbox={"facecolor": BACKGROUND_COLOR, "edgecolor": "none", "pad": 1.5},
    )
    ax.margins(x=0.04, y=0.15)
    ax.legend(
        loc="upper right", bbox_to_anchor=(0.99, 0.99), frameon=False,
        fontsize=LEGEND_SIZE, handlelength=2.2, labelcolor=TEXT_COLOR,
    )
    ax.set_xlabel("경과 연차", color=TICK_COLOR, fontsize=AXIS_TITLE_SIZE, labelpad=10)
    ax.set_ylabel("잔액(억원)", color=TICK_COLOR, fontsize=AXIS_TITLE_SIZE, labelpad=10)

    fig.text(0.02, 0.99, "대도시 연구실", ha="left", va="top", fontsize=BRAND_SIZE, color=SECONDARY_TEXT_COLOR)
    fig.suptitle(
        f"인출순서에 따른 {INVESTMENT_YEARS}년 잔액 변화",
        x=0.02, y=0.95, ha="left", va="top", fontsize=TITLE_SIZE, fontweight="bold", color=TEXT_COLOR,
    )
    fig.text(
        0.02, 0.885,
        f"연 {연상승률_퍼센트:g}% 상승(월 복리) 가정 · "
        f"초기 준비금 {format_korean_won(INITIAL_RESERVE_KRW).replace('억원', '억 원')} · "
        f"월 {format_korean_won(MONTHLY_NET_WITHDRAWAL_KRW).replace('만원', '만 원')} 인출 · "
        f"{INVESTOR_LABEL} 기준",
        ha="left", va="top", fontsize=SUBTITLE_SIZE, color=SECONDARY_TEXT_COLOR,
    )
    fig.text(
        0.02, 0.01,
        "가상 데이터: 연상승률을 월 복리로 환산해 매달 동일한 비율로 상승한다고 가정\n"
        "※ 두 전략은 원금성 재원의 인출 순서가 같아 잔액 곡선이 초반에는 거의 겹치며, "
        "공제분·연금 수익금·ISA 해지 순서가 갈리는 후반부에서 벌어진다",
        va="bottom", fontsize=FOOTNOTE_SIZE, color=SECONDARY_TEXT_COLOR,
    )
    fig.subplots_adjust(left=0.08, right=0.97, bottom=0.14, top=0.8)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor=BACKGROUND_COLOR)
    plt.close(fig)


def save_monthly_balance_csv(detail: pd.DataFrame, output_dir: Path) -> dict[str, Path]:
    """전략별 계좌별 월말 잔액을 원금·수익금으로 나눠 저장한다."""
    paths: dict[str, Path] = {}
    for strategy in STRATEGIES:
        flow = detail.loc[detail["strategy"] == strategy].sort_values("month_number")
        balance = pd.DataFrame({
            "경과월": flow["month_number"],
            "미국계좌 공제분": flow["us_core_value_krw"],
            "ISA원금": flow["isa_principal_krw"],
            "ISA수익금": flow["isa_gain_krw"],
            "연금저축원금": flow["pension_principal_krw"],
            "연금저축수익금": flow["pension_gain_krw"],
            "국내일반계좌": flow["general_krw"],
            "미국계좌 과세분": flow["us_excess_krw"],
            "총평가액": flow["ending_balance_krw"],
            "당월실수령액": flow["actual_withdrawal_krw"],
            "당월납부세금": flow["tax_paid_krw"],
        })
        balance_path = output_dir / f"st_tax_withdrawal_{strategy}_monthly_balance.csv"
        numeric_columns = balance.columns.drop("경과월")
        balance[numeric_columns] = balance[numeric_columns].round(0).astype("int64")
        balance.to_csv(balance_path, index=False, encoding="utf-8-sig")
        paths[strategy] = balance_path
    return paths


def save_results(summary: pd.DataFrame, detail: pd.DataFrame, output_dir: Path) -> dict[str, Any]:
    """요약·월별 상세·전략별 월별 잔액 CSV와 잔액 흐름 그래프를 저장한다."""
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = output_dir / "st_tax_withdrawal_summary.csv"
    detail_path = output_dir / "st_tax_withdrawal_monthly_detail.csv"
    flow_path = output_dir / "st_tax_withdrawal_balance_flow.png"
    summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
    detail.to_csv(detail_path, index=False, encoding="utf-8-sig")
    balance_paths = save_monthly_balance_csv(detail, output_dir)
    save_balance_flow_chart(summary, detail, flow_path)
    return {
        "summary": summary_path, "detail": detail_path,
        "balance": balance_paths, "flow": flow_path,
    }


def display_results(summary: pd.DataFrame, paths: dict[str, Any]) -> None:
    """핵심 조건, 그래프와 저장 경로를 표시한다."""
    display(Markdown(
        "## 인출 전략별 결과\n\n"
        + "\n".join(
            f"- **{strategy}**: "
            + (
                "생존"
                if not bool(cast(Any, summary.loc[summary['strategy'] == strategy].iloc[0]).depleted_within_period)
                else (
                    "고갈 ("
                    f"{format_duration(int(cast(Any, summary.loc[summary['strategy'] == strategy].iloc[0]).depletion_month_number))}"
                    " 후)"
                )
            )
            for strategy in STRATEGIES
        )
    ))
    display(HTML(result_table_html(summary)))
    display(Image(filename=str(paths["flow"])))
    print("저장 완료:")
    print(paths["summary"])
    print(paths["detail"])
    for strategy, path in paths["balance"].items():
        print(f"{strategy}: {path}")
    print(paths["flow"])


def main() -> None:
    """입력 검증부터 가상 데이터 생성, 계산, 저장과 출력까지 실행한다."""
    validate_parameters()
    market = build_virtual_market()
    summary, detail = calculate_results(market)
    paths = save_results(summary, detail, OUTPUT_DIR)
    display_results(summary, paths)


main()


In [ ]:
# @title 연차별 절세 계좌 배분 다이어그램을 그리려면 이 셀을 실행하세요
"""인출 없이 초기 준비금만 절세 계좌에 배분했을 때 연차별 자금 흐름을 그린다.

위 인출 시뮬레이션과는 무관한 별도 계산이다. 1년차에 초기 준비금을
"미국계좌 공제분 → ISA → 연금저축 → 국내 일반계좌 → 미국계좌 과세분"
순으로 배분한 뒤, 매년 모든 계좌가 ANNUAL_GROWTH_RATE만큼 복리 성장한다.

연말에는 두 계좌가 한도 초과분을 리밸런싱한다. 미국계좌 공제분은 US_
DEDUCTION_KRW를, 국내 일반계좌는 GENERAL_MAX_KRW를 초과하면 초과분을
매도해 국내 일반계좌는 GENERAL_TAX_RATE로 과세한 뒤(미국계좌 공제분은
비과세) 세후 금액을 연금저축 남은 연간 납입한도 → 미국계좌 과세분
순으로 옮긴다. 연초에는 연금저축 연간 납입한도 중 남은 여유만큼 미국
계좌 과세분 → 국내 일반계좌 순으로 재원을 조달해 추가 납입한다. ISA는
1년차 납입 이후 추가 납입이 없다. 인출은 반영하지 않는다.
"""

from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

ALLOCATION_YEARS = 5

ALLOCATION_COLUMN_KEYS = ["us_core", "isa", "pension", "general", "us_taxable"]
ALLOCATION_COLUMN_COLORS = {
    "us_core": "#2F7DD3",
    "isa": "#1FAE7A",
    "pension": "#F0A93A",
    "general": "#DC5B4B",
    "us_taxable": "#94A3B8",
}
ALLOCATION_COLUMN_LABELS = {
    "us_core": "미국계좌 공제분", "isa": "ISA", "pension": "연금저축",
    "general": "국내 일반계좌", "us_taxable": "미국계좌 과세분",
}


def format_signed_won(amount_krw: float) -> str:
    """부호를 붙인 원화 금액을 억원·만원 단위로 표시한다."""
    sign = "+" if amount_krw >= 0 else "-"
    return f"{sign}{format_korean_won(abs(amount_krw))}"


def compute_allocation_flow() -> list[dict[str, Any]]:
    """연차별 절세 계좌 배분 흐름(원금·수익 분리)을 계산한다."""
    remaining_krw = INITIAL_RESERVE_KRW
    us_core_krw = min(remaining_krw, US_CORE_KRW)
    remaining_krw -= us_core_krw
    isa_principal_krw = min(remaining_krw, ISA_MAX_KRW)
    remaining_krw -= isa_principal_krw
    isa_gain_krw = 0.0
    pension_principal_krw = min(remaining_krw, PENSION_ANNUAL_LIMIT_KRW)
    remaining_krw -= pension_principal_krw
    pension_gain_krw = 0.0
    general_krw = min(remaining_krw, GENERAL_MAX_KRW)
    remaining_krw -= general_krw
    us_taxable_krw = remaining_krw

    # 1년차 초기 배분으로 그해 연금저축 납입한도를 이미 소모한 것으로 취급한다.
    pension_annual_contributed_krw = pension_principal_krw

    rows: list[dict[str, Any]] = []
    for year in range(1, ALLOCATION_YEARS + 1):
        if year > 1:
            pension_annual_contributed_krw = 0.0

            # 1) 모든 계좌에 1년치 성장을 반영한다.
            us_core_krw *= (1 + ANNUAL_GROWTH_RATE)
            general_krw *= (1 + ANNUAL_GROWTH_RATE)
            isa_gain_krw = (
                (isa_principal_krw + isa_gain_krw) * (1 + ANNUAL_GROWTH_RATE) - isa_principal_krw
            )
            pension_gain_krw = (
                (pension_principal_krw + pension_gain_krw) * (1 + ANNUAL_GROWTH_RATE)
                - pension_principal_krw
            )
            us_taxable_krw *= (1 + ANNUAL_GROWTH_RATE)

            # 2) 국내 일반계좌 한도 초과분을 매도·과세한 뒤 재배분한다.
            if general_krw > GENERAL_MAX_KRW:
                excess_krw = general_krw - GENERAL_MAX_KRW
                general_krw = GENERAL_MAX_KRW
                tax_krw = excess_krw * GENERAL_TAX_RATE
                net_krw = excess_krw - tax_krw

                pension_room_krw = max(
                    PENSION_ANNUAL_LIMIT_KRW - pension_annual_contributed_krw, 0.0
                )
                from_general_to_pension_krw = min(pension_room_krw, net_krw)
                pension_principal_krw += from_general_to_pension_krw
                pension_annual_contributed_krw += from_general_to_pension_krw
                net_krw -= from_general_to_pension_krw

                us_taxable_krw += net_krw

            # 3) 미국계좌 공제분 한도 초과분을 비과세로 재배분한다.
            if us_core_krw > US_CORE_KRW:
                excess_krw = us_core_krw - US_CORE_KRW
                us_core_krw = US_CORE_KRW
                us_taxable_krw += excess_krw

            # 4) 연초 연금저축 추가 납입: 남은 여유만큼 미국계좌 과세분 → 국내 일반계좌 순으로 조달한다.
            needed_krw = max(PENSION_ANNUAL_LIMIT_KRW - pension_annual_contributed_krw, 0.0)
            from_us_taxable_krw = min(us_taxable_krw, needed_krw)
            us_taxable_krw -= from_us_taxable_krw
            needed_krw -= from_us_taxable_krw
            from_general_krw = min(general_krw, needed_krw)
            general_krw -= from_general_krw
            needed_krw -= from_general_krw
            contributed_krw = (
                (PENSION_ANNUAL_LIMIT_KRW - pension_annual_contributed_krw) - needed_krw
            )
            pension_principal_krw += contributed_krw
            pension_annual_contributed_krw += contributed_krw

        rows.append({
            "year": year, "us_core": us_core_krw,
            "isa_principal": isa_principal_krw, "isa_gain": isa_gain_krw,
            "pension_principal": pension_principal_krw, "pension_gain": pension_gain_krw,
            "general": general_krw, "us_taxable": us_taxable_krw,
            "isa_total": isa_principal_krw + isa_gain_krw,
            "pension_total": pension_principal_krw + pension_gain_krw,
        })
    return rows


def allocation_total_value(row: dict[str, Any], key: str) -> float:
    """열의 합계값(ISA·연금저축은 원금+수익)을 반환한다."""
    if key == "isa":
        return row["isa_total"]
    if key == "pension":
        return row["pension_total"]
    return row[key]


def save_allocation_flow_diagram(rows: list[dict[str, Any]], output_path: Path) -> None:
    """연차별 절세 계좌 배분 흐름을 박스·화살표 다이어그램으로 저장한다."""
    configure_korean_font()
    n_years = len(rows)
    n_cols = len(ALLOCATION_COLUMN_KEYS)
    col_width = 1.35
    col_gap = 0.55
    row_height = 0.66
    row_gap = 0.5

    fig_width = n_cols * (col_width + col_gap) + 1.2
    fig_height = n_years * (row_height + row_gap) + 2.4
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    fig.patch.set_facecolor(BACKGROUND_COLOR)
    ax.set_facecolor(BACKGROUND_COLOR)

    def col_x(i: int) -> float:
        return i * (col_width + col_gap)

    def year_top_y(year: int) -> float:
        return (n_years - year) * (row_height + row_gap)

    header_y = year_top_y(1) + row_height + 0.35
    for i, key in enumerate(ALLOCATION_COLUMN_KEYS):
        x = col_x(i) + col_width / 2
        ax.text(x, header_y, ALLOCATION_COLUMN_LABELS[key], ha="center", va="bottom",
                fontsize=13.5, fontweight="bold", color=TEXT_COLOR)

    split_gap = 0.04
    for idx, row in enumerate(rows):
        y = year_top_y(row["year"])
        ax.text(-0.35, y + row_height / 2, f"{row['year']}년차", ha="right", va="center",
                fontsize=12.5, fontweight="bold", color=TEXT_COLOR)

        for i, key in enumerate(ALLOCATION_COLUMN_KEYS):
            x = col_x(i)
            color = ALLOCATION_COLUMN_COLORS[key]
            if key in ("isa", "pension"):
                principal_krw = row[f"{key}_principal"]
                gain_krw = row[f"{key}_gain"]
                half_h = (row_height - split_gap) / 2
                box_principal = FancyBboxPatch(
                    (x, y + half_h + split_gap), col_width, half_h,
                    boxstyle="round,pad=0,rounding_size=0.045",
                    linewidth=1.3, edgecolor=color, facecolor=color + "2E",
                )
                ax.add_patch(box_principal)
                ax.text(x + col_width / 2, y + half_h + split_gap + half_h / 2,
                        f"원금 {format_korean_won(principal_krw)}", ha="center", va="center",
                        fontsize=12, fontweight="bold", color=color)
                box_gain = FancyBboxPatch(
                    (x, y), col_width, half_h,
                    boxstyle="round,pad=0,rounding_size=0.045",
                    linewidth=1.0, edgecolor=color, facecolor=color + "14",
                )
                ax.add_patch(box_gain)
                ax.text(x + col_width / 2, y + half_h / 2,
                        f"수익 {format_korean_won(gain_krw)}", ha="center", va="center",
                        fontsize=12, fontweight="bold", color=color)
            else:
                value_krw = row[key]
                box = FancyBboxPatch(
                    (x, y), col_width, row_height,
                    boxstyle="round,pad=0,rounding_size=0.05",
                    linewidth=1.3, edgecolor=color, facecolor=color + "22",
                )
                ax.add_patch(box)
                ax.text(x + col_width / 2, y + row_height / 2, format_korean_won(value_krw),
                        ha="center", va="center", fontsize=13.5, fontweight="bold", color=color)

            if idx < n_years - 1:
                next_row = rows[idx + 1]
                delta_krw = allocation_total_value(next_row, key) - allocation_total_value(row, key)
                arrow_x = x + col_width / 2
                arrow_top = y - 0.02
                arrow_bottom = y - (row_gap - 0.06)
                arrow = FancyArrowPatch(
                    (arrow_x, arrow_top), (arrow_x, arrow_bottom),
                    arrowstyle="-|>", mutation_scale=11,
                    linewidth=1.1, color=SECONDARY_TEXT_COLOR, zorder=2,
                )
                ax.add_patch(arrow)
                if abs(delta_krw) >= WON_PER_MANWON * 0.5:
                    delta_color = "#1FAE7A" if delta_krw > 0 else "#DC5B4B"
                    ax.text(arrow_x + 0.09, (arrow_top + arrow_bottom) / 2, format_signed_won(delta_krw),
                            ha="left", va="center", fontsize=11.5, color=delta_color)

    ax.set_xlim(-0.7, col_x(n_cols - 1) + col_width + 0.3)
    ax.set_ylim(0.0, header_y + 0.4)
    ax.axis("off")
    fig.text(0.02, 0.995, "대도시 연구실", ha="left", va="top", fontsize=11.5, color=SECONDARY_TEXT_COLOR)
    fig.suptitle(
        f"{format_korean_won(INITIAL_RESERVE_KRW)}의 연차별 절세 계좌 배분",
        x=0.02, y=0.965, ha="left", va="top", fontsize=19, fontweight="bold", color=TEXT_COLOR,
    )
    fig.text(
        0.02, 0.93,
        f"{INVESTOR_LABEL} · 연 {연상승률_퍼센트:g}% 수익률 · 인출 없이 순수 배분만 · 금액은 각 연도 시작 기준",
        ha="left", va="top", fontsize=12, color=SECONDARY_TEXT_COLOR,
    )
    fig.text(
        0.02, -0.01,
        "※ 1년차 배분 순서: 미국계좌 공제분 2,500만원 → ISA 1억원 → 연금저축 1,800만원 → 국내 일반계좌 2억원 → 미국계좌 과세분\n"
        "※ 매년 말 미국계좌 공제분 2,500만원·국내 일반계좌 2억원 초과분을 매도(국내 일반계좌만 15.4% 과세)해 연금저축 잔여 납입한도 → 미국계좌 과세분 순으로 이전\n"
        "※ 매년 초 연금저축 1,800만원 잔여 납입한도는 미국계좌 과세분 → 국내 일반계좌 순으로 조달",
        ha="left", va="bottom", fontsize=11.5, color=SECONDARY_TEXT_COLOR,
    )
    fig.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.06)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor=BACKGROUND_COLOR)
    plt.close(fig)


allocation_rows = compute_allocation_flow()
allocation_flow_path = OUTPUT_DIR / "st_tax_withdrawal_account_allocation_flow.png"
save_allocation_flow_diagram(allocation_rows, allocation_flow_path)
display(Image(filename=str(allocation_flow_path)))
print("저장 완료:")
print(allocation_flow_path)


In [ ]:
# @title 과세이연 인출 시나리오의 연차별 계좌 배분 다이어그램을 그리려면 이 셀을 실행하세요
"""위 인출 시뮬레이션(과세이연 전략)의 계좌 잔액 흐름을 그린다.

simulate_withdrawal_strategy를 그대로 사용해 월 단위로 계산한다. 1~2년차는
매년 시작 시점(전년 12월 리밸런싱·연초 연금저축 납입 반영 직후) 스냅샷을
그대로 보여준다. 3년차부터는 정기 연차 스냅샷 대신, 각 계좌가 0원이 되는
모든 시점(같은 계좌가 ISA 해지 등으로 재충전된 뒤 다시 소진되는 두 번째
고갈까지 포함)을 순서대로 포착해 보여준다. 같은 달에 여러 이벤트가 겹치면
한 행으로 합친다.
"""

WITHDRAWAL_STRATEGY_FOR_FLOW = "과세이연"
WITHDRAWAL_FLOW_REGULAR_YEARS = 2  # 이 연차까지는 정기 연차 스냅샷을 그대로 사용


def format_year_month_label(month_number: int) -> str:
    """경과 개월수를 "N년차" 또는 "N년 M개월차"로 표시한다."""
    years, months = divmod(month_number, MONTHS_PER_YEAR)
    if months == 0:
        return f"{years}년차"
    return f"{years}년 {months}개월차"


EVENT_FIELD_TO_BOX_KEY = {
    "us_excess_krw": "us_taxable", "general_krw": "general",
    "pension_principal_krw": "pension_principal", "isa_principal_krw": "isa_principal",
    "us_core_value_krw": "us_core", "pension_gain_krw": "pension_gain",
    "isa_gain_krw": "isa_gain",
}
# ISA 수익금이 소진되는 시점은 ISA 만기해지가 일어나는 시점과 같다.
ISA_CLOSE_BOX_KEY = "isa_gain"


def snapshot_to_row(
    r: Any, label: str, depleted_box_keys: set[str], isa_closed_here: bool
) -> dict[str, Any]:
    """detail 한 행을 다이어그램용 계좌 스냅샷으로 변환한다."""
    return {
        "label": label,
        "us_core": float(r.us_core_value_krw),
        "isa_principal": float(r.isa_principal_krw), "isa_gain": float(r.isa_gain_krw),
        "pension_principal": float(r.pension_principal_krw), "pension_gain": float(r.pension_gain_krw),
        "general": float(r.general_krw), "us_taxable": float(r.us_excess_krw),
        "isa_total": float(r.isa_principal_krw) + float(r.isa_gain_krw),
        "pension_total": float(r.pension_principal_krw) + float(r.pension_gain_krw),
        "depleted_box_keys": depleted_box_keys,
        "isa_closed_here": isa_closed_here,
    }


def compute_withdrawal_allocation_flow(strategy: str) -> list[dict[str, Any]]:
    """인출 시뮬레이션에서 정기 연차 스냅샷(초반)과 계좌별 고갈 이벤트 스냅샷(전체)을 뽑는다."""
    market = build_virtual_market()
    _, detail = simulate_withdrawal_strategy(market, strategy)
    detail = detail.sort_values("month_number").reset_index(drop=True)
    last_month = int(detail.iloc[-1]["month_number"])

    rows: list[dict[str, Any]] = []
    shown_months: set[int] = set()
    regular_last_month = WITHDRAWAL_FLOW_REGULAR_YEARS * MONTHS_PER_YEAR

    # 1) 초반 정기 연차 스냅샷 (1년차, 2년차, ...)
    for year in range(1, WITHDRAWAL_FLOW_REGULAR_YEARS + 1):
        snapshot_month = (year - 1) * MONTHS_PER_YEAR
        if snapshot_month > last_month:
            break
        snapshot = detail.loc[detail["month_number"] == snapshot_month]
        if snapshot.empty:
            continue
        r = cast(Any, snapshot.iloc[0])
        rows.append(snapshot_to_row(r, f"{year}년차", set(), False))
        shown_months.add(snapshot_month)

    # 2) 계좌별로 값이 0을 향해 처음 진입하는 모든 시점을 순서대로 탐지한다.
    #    (ISA 해지처럼 재충전 후 다시 소진되는 두 번째 고갈도 포함한다.)
    event_fields = [
        "us_excess_krw", "general_krw", "pension_principal_krw",
        "isa_principal_krw", "us_core_value_krw", "pension_gain_krw", "isa_gain_krw",
    ]
    fields_using_next_month = {"general_krw"}
    # 미국계좌 과세분은 인출 순서 1순위라 매년 12월 리밸런싱으로 채워졌다가
    # 다음 인출로 바로 소진되는 패턴이 반복된다. 이 정상적인 반복은 이벤트로
    # 잡지 않고, 처음으로 0이 되는 시점만 이벤트로 남긴다.
    fields_first_zero_only = {"us_excess_krw"}
    max_available_month = int(detail["month_number"].max())
    # regular_last_month 시점 값을 prev_value의 시작점으로 포함해야, 그 달
    # 이후 바로 0이 되는 진짜 첫 전환을 놓치지 않는다(이 달 자체는 이벤트로
    # 채택하지 않는다).
    after_regular = detail.loc[detail["month_number"] >= regular_last_month].reset_index(drop=True)

    month_to_depleted_keys: dict[int, set[str]] = {}
    isa_close_months: set[int] = set()
    for field in event_fields:
        prev_value = None
        for _, r in after_regular.iterrows():
            value = float(r[field])
            month_number = int(r["month_number"])
            if month_number == regular_last_month:
                prev_value = value
                continue
            if prev_value is not None and prev_value > 0.5 and value <= 0.5:
                event_month = month_number
                if field in fields_using_next_month and event_month < max_available_month:
                    event_month += 1
                month_to_depleted_keys.setdefault(event_month, set()).add(
                    EVENT_FIELD_TO_BOX_KEY[field]
                )
                if field == "isa_gain_krw":
                    isa_close_months.add(event_month)
                if field in fields_first_zero_only:
                    break
            prev_value = value

    # 3) 전액 소진 시점(마지막 달)도 이벤트로 포함한다. 마지막 재원(미국계좌
    #    공제분)이 사실상 소진 단계에 들어선 직전 이벤트 행에도 고갈 표시를
    #    함께 남긴다.
    prior_event_months = [m for m in month_to_depleted_keys if m < last_month]
    if prior_event_months:
        final_depletion_month = max(prior_event_months)
        month_to_depleted_keys[final_depletion_month] = (
            month_to_depleted_keys.get(final_depletion_month, set()) | {"us_core"}
        )
    month_to_depleted_keys.setdefault(last_month, set()).add("us_core")
    all_event_months = sorted(month_to_depleted_keys.keys())
    all_event_months = [m for m in all_event_months if m > regular_last_month]

    for month_number in all_event_months:
        if month_number in shown_months:
            continue
        snapshot = detail.loc[detail["month_number"] == month_number]
        if snapshot.empty:
            continue
        r = cast(Any, snapshot.iloc[0])
        depleted_box_keys = month_to_depleted_keys.get(month_number, set())
        rows.append(snapshot_to_row(
            r, format_year_month_label(month_number), depleted_box_keys,
            month_number in isa_close_months,
        ))
        shown_months.add(month_number)

    return rows


def save_withdrawal_allocation_flow_diagram(
    rows: list[dict[str, Any]], strategy: str, output_path: Path
) -> None:
    """인출 시나리오의 계좌 배분 흐름을 박스·화살표 다이어그램으로 저장한다."""
    configure_korean_font()
    n_rows = len(rows)
    n_cols = len(ALLOCATION_COLUMN_KEYS)
    col_width = 1.35
    col_gap = 0.55
    row_height = 0.66
    row_gap = 0.5

    fig_width = n_cols * (col_width + col_gap) + 1.4
    fig_height = n_rows * (row_height + row_gap) + 2.4
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    fig.patch.set_facecolor(BACKGROUND_COLOR)
    ax.set_facecolor(BACKGROUND_COLOR)

    def col_x(i: int) -> float:
        return i * (col_width + col_gap)

    def row_top_y(row_index: int) -> float:
        return (n_rows - 1 - row_index) * (row_height + row_gap)

    header_y = row_top_y(0) + row_height + 0.35
    for i, key in enumerate(ALLOCATION_COLUMN_KEYS):
        x = col_x(i) + col_width / 2
        ax.text(x, header_y, ALLOCATION_COLUMN_LABELS[key], ha="center", va="bottom",
                fontsize=13.5, fontweight="bold", color=TEXT_COLOR)

    split_gap = 0.04
    DEPLETION_BADGE_COLOR = "#DC2626"

    def draw_status_badge(
        box_x: float, box_y: float, box_w: float, box_h: float, text: str
    ) -> None:
        badge_w = 0.38 if text == "고갈" else 0.42
        badge_h = 0.17
        badge_x = box_x + box_w - badge_w - 0.03
        badge_y = box_y + 0.03
        ax.add_patch(FancyBboxPatch(
            (badge_x, badge_y), badge_w, badge_h,
            boxstyle="round,pad=0,rounding_size=0.03",
            linewidth=0, facecolor=DEPLETION_BADGE_COLOR, zorder=4,
        ))
        ax.text(badge_x + badge_w / 2, badge_y + badge_h / 2, text,
                ha="center", va="center", fontsize=9.5, fontweight="bold", color="white", zorder=5)

    for idx, row in enumerate(rows):
        y = row_top_y(idx)
        ax.text(-0.45, y + row_height / 2, row["label"], ha="right", va="center",
                fontsize=12.5, fontweight="bold", color=TEXT_COLOR)

        for i, key in enumerate(ALLOCATION_COLUMN_KEYS):
            x = col_x(i)
            color = ALLOCATION_COLUMN_COLORS[key]
            if key in ("isa", "pension"):
                principal_krw = row[f"{key}_principal"]
                gain_krw = row[f"{key}_gain"]
                half_h = (row_height - split_gap) / 2
                box_principal = FancyBboxPatch(
                    (x, y + half_h + split_gap), col_width, half_h,
                    boxstyle="round,pad=0,rounding_size=0.045",
                    linewidth=1.3, edgecolor=color, facecolor=color + "2E",
                )
                ax.add_patch(box_principal)
                ax.text(x + col_width / 2, y + half_h + split_gap + half_h / 2,
                        f"원금 {format_korean_won(principal_krw)}", ha="center", va="center",
                        fontsize=12, fontweight="bold", color=color)
                if f"{key}_principal" in row["depleted_box_keys"]:
                    draw_status_badge(x, y + half_h + split_gap, col_width, half_h, "고갈")
                box_gain = FancyBboxPatch(
                    (x, y), col_width, half_h,
                    boxstyle="round,pad=0,rounding_size=0.045",
                    linewidth=1.0, edgecolor=color, facecolor=color + "14",
                )
                ax.add_patch(box_gain)
                ax.text(x + col_width / 2, y + half_h / 2,
                        f"수익 {format_korean_won(gain_krw)}", ha="center", va="center",
                        fontsize=12, fontweight="bold", color=color)
                if f"{key}_gain" in row["depleted_box_keys"]:
                    badge_text = "해지" if (key == "isa" and row["isa_closed_here"]) else "고갈"
                    draw_status_badge(x, y, col_width, half_h, badge_text)
            else:
                value_krw = row[key]
                box = FancyBboxPatch(
                    (x, y), col_width, row_height,
                    boxstyle="round,pad=0,rounding_size=0.05",
                    linewidth=1.3, edgecolor=color, facecolor=color + "22",
                )
                ax.add_patch(box)
                ax.text(x + col_width / 2, y + row_height / 2, format_korean_won(value_krw),
                        ha="center", va="center", fontsize=13.5, fontweight="bold", color=color)
                if key in row["depleted_box_keys"]:
                    draw_status_badge(x, y, col_width, row_height, "고갈")

            if idx < n_rows - 1:
                next_row = rows[idx + 1]
                delta_krw = allocation_total_value(next_row, key) - allocation_total_value(row, key)
                arrow_x = x + col_width / 2
                arrow_top = y - 0.02
                arrow_bottom = y - (row_gap - 0.06)
                arrow = FancyArrowPatch(
                    (arrow_x, arrow_top), (arrow_x, arrow_bottom),
                    arrowstyle="-|>", mutation_scale=11,
                    linewidth=1.1, color=SECONDARY_TEXT_COLOR, zorder=2,
                )
                ax.add_patch(arrow)
                if abs(delta_krw) >= WON_PER_MANWON * 0.5:
                    delta_color = "#1FAE7A" if delta_krw > 0 else "#DC5B4B"
                    ax.text(arrow_x + 0.09, (arrow_top + arrow_bottom) / 2, format_signed_won(delta_krw),
                            ha="left", va="center", fontsize=11.5, color=delta_color)

    ax.set_xlim(-1.1, col_x(n_cols - 1) + col_width + 0.3)
    ax.set_ylim(0.0, header_y + 0.4)
    ax.axis("off")
    fig.text(0.02, 0.995, "대도시 연구실", ha="left", va="top", fontsize=11.5, color=SECONDARY_TEXT_COLOR)
    fig.suptitle(
        f"{format_korean_won(INITIAL_RESERVE_KRW)}의 연차별 절세 계좌 배분 · {strategy} 인출",
        x=0.02, y=0.985, ha="left", va="top", fontsize=19, fontweight="bold", color=TEXT_COLOR,
    )
    fig.text(
        0.02, 0.958,
        f"{INVESTOR_LABEL} · 연 {연상승률_퍼센트:g}% 수익률 · "
        f"월 {format_korean_won(MONTHLY_NET_WITHDRAWAL_KRW).replace('만원', '만 원')} 인출 · "
        f"{WITHDRAWAL_FLOW_REGULAR_YEARS}년차까지는 매년 시작 기준, 이후는 계좌별 소진 시점 기준",
        ha="left", va="top", fontsize=12, color=SECONDARY_TEXT_COLOR,
    )
    fig.text(
        0.02, 0.028,
        "※ 1년차 배분 순서: 미국계좌 공제분 2,500만원 → ISA 1억원 → 연금저축 1,800만원 → 국내 일반계좌 2억원 → 미국계좌 과세분\n"
        "※ 과세이연 인출 순서: 미국계좌 과세분 → 국내 일반계좌 → 연금저축 원금 → ISA 원금 → 미국계좌 공제분 → 연금저축 수익금 → ISA 해지",
        ha="left", va="bottom", fontsize=11.5, color=SECONDARY_TEXT_COLOR,
    )
    fig.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.09)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor=BACKGROUND_COLOR)
    plt.close(fig)


withdrawal_allocation_rows = compute_withdrawal_allocation_flow(WITHDRAWAL_STRATEGY_FOR_FLOW)
withdrawal_allocation_flow_path = (
    OUTPUT_DIR / "st_tax_withdrawal_account_allocation_flow_과세이연_인출.png"
)
save_withdrawal_allocation_flow_diagram(
    withdrawal_allocation_rows, WITHDRAWAL_STRATEGY_FOR_FLOW, withdrawal_allocation_flow_path
)
display(Image(filename=str(withdrawal_allocation_flow_path)))
print("저장 완료:")
print(withdrawal_allocation_flow_path)
print(f"표시된 행 수: {len(withdrawal_allocation_rows)}")
for row in withdrawal_allocation_rows:
    print(row["label"], row["depleted_box_keys"], "ISA해지" if row["isa_closed_here"] else "")
